# 02 Embedding Review

**Goal:** Validate the quality of chunk embeddings before clustering.

This notebook checks:
- Whether all chunks have embeddings and none are missing
- Embedding dimensions are consistent (1024-dim, BAAI/bge-large-en-v1.5)
- No duplicate embedding texts across chunks
- Whether embeddings are semantically meaningful — using cosine similarity to retrieve nearest neighbours for sample chunks and verify that similar chunks are being grouped correctly


In [19]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

EMBEDDINGS_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "chunk_embeddings"
    / "chunk_embeddings_flat.json"
)

with open(EMBEDDINGS_PATH, "r", encoding="utf-8") as f:
    flat_chunks = json.load(f)

emb_df = pd.DataFrame(flat_chunks)

emb_df.head()

,meeting_id,meeting_title,chunk_id,start_turn,end_turn,start_time,end_time,chunk_label,chunk_summary,chunk_text,embedding_text,chunk_sentiment,sentiment_reason,turn_sentiment_counts,matched_provided_topics,is_cross_domain,call_type,invitee_domains,embedding_model,embedding
0,01KQ03B0303900521BB089CA,Detect Outage - Remediation Plan Review,01KQ03B0303900521BB089CA_CHUNK_001,0,3,7.4,41.0,Outage Overview,Team discusses current outage status and need ...,"Megan Lawson: Alright, I think we're all on — ...",Outage Overview. Team discusses current outage...,neutral,Initial discussion focuses on aligning on the ...,{'neutral': 4},[incident response],False,internal_operational,[aegiscloud.com],BAAI/bge-large-en-v1.5,"[-0.0076064495369791985, -0.012960203923285007..."
1,01KQ03B0303900521BB089CA,Detect Outage - Remediation Plan Review,01KQ03B0303900521BB089CA_CHUNK_002,4,15,42.3,210.0,Root Cause Analysis,Agent explains the cascading failure and propo...,"Raj Kapoor: Yeah, agreed. I've been heads-down...",Root Cause Analysis. Agent explains the cascad...,neutral,Technical details shared about the outage's ro...,"{'neutral': 10, 'negative': 2}",[outage remediation],False,internal_operational,[aegiscloud.com],BAAI/bge-large-en-v1.5,"[-0.03413614258170128, -0.0030017392709851265,..."
2,01KQ03B0303900521BB089CA,Detect Outage - Remediation Plan Review,01KQ03B0303900521BB089CA_CHUNK_003,16,20,210.6,266.4,Customer Communication,Discussion on updating customers about visibil...,Brian Cho: Okay so when you say resolved today...,Customer Communication. Discussion on updating...,mixed-negative,Concerns raised about managing customer expect...,"{'neutral': 3, 'negative': 2}",[customer communication],False,internal_operational,[aegiscloud.com],BAAI/bge-large-en-v1.5,"[0.004239668603986502, -0.01763724535703659, -..."
3,01KQ03B0303900521BB089CA,Detect Outage - Remediation Plan Review,01KQ03B0303900521BB089CA_CHUNK_004,21,30,267.5,394.4,Security Assurance,Team discusses threat monitoring data during o...,Megan Lawson: I can help draft the updated cus...,Security Assurance. Team discusses threat moni...,neutral,Clarification on security posture and ongoing ...,"{'neutral': 9, 'negative': 1}",[security monitoring],False,internal_operational,[aegiscloud.com],BAAI/bge-large-en-v1.5,"[-0.015702230855822563, 0.006529663689434528, ..."
4,01KQ03B0303900521BB089CA,Detect Outage - Remediation Plan Review,01KQ03B0303900521BB089CA_CHUNK_005,31,42,395.0,515.4,Postmortem Planning,Aligning on postmortem process and data needed...,Megan Lawson: Alright. Let's talk about the po...,Postmortem Planning. Aligning on postmortem pr...,neutral,Discussion on internal and external postmortem...,{'neutral': 12},[postmortem planning],False,internal_operational,[aegiscloud.com],BAAI/bge-large-en-v1.5,"[0.028703007847070694, -0.0053783864714205265,..."


In [13]:
print("Total chunks:", len(emb_df))
print("Missing embeddings:", emb_df["embedding"].isna().sum())
print("Missing embedding_text:", emb_df["embedding_text"].isna().sum())
print("Unique meetings:", emb_df["meeting_id"].nunique())
print("Unique chunk labels:", emb_df["chunk_label"].nunique())

Total chunks: 84
Missing embeddings: 0
Missing embedding_text: 0
Unique meetings: 20
Unique chunk labels: 83


In [14]:
embedding_matrix = np.array(emb_df["embedding"].tolist())

embedding_matrix.shape

(84, 1024)

In [15]:
def show_nearest_chunks(row_index, top_k=5):
    query_vector = embedding_matrix[row_index].reshape(1, -1)

    similarities = cosine_similarity(query_vector, embedding_matrix)[0]

    nearest_indices = similarities.argsort()[::-1][1:top_k + 1]

    cols = [
        "meeting_id",
        "chunk_id",
        "chunk_label",
        "chunk_summary",
        "chunk_sentiment",
    ]

    result = emb_df.iloc[nearest_indices][cols].copy()
    result["similarity"] = similarities[nearest_indices]

    return result


show_nearest_chunks(row_index=0, top_k=5)

/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,meeting_id,chunk_id,chunk_label,chunk_summary,chunk_sentiment,similarity
77,01KQ351E141926AB7CAB668D,01KQ351E141926AB7CAB668D_CHUNK_001,Outage Investigation,"Agent explains outage, root cause, regulatory ...",negative,0.893177
44,01KQ2217B066855A3B7814CB,01KQ2217B066855A3B7814CB_CHUNK_002,Outage Incident Overview,"Agent details outage, lack of threat monitorin...",negative,0.876101
36,01KQ1DCC80852AE384C898C9,01KQ1DCC80852AE384C898C9_CHUNK_003,Remediation Documentation,"Agent explains root cause, offers technical re...",neutral,0.867282
2,01KQ03B0303900521BB089CA,01KQ03B0303900521BB089CA_CHUNK_003,Remediation Plan,"Agent details redundant nodes, circuit breaker...",neutral,0.852308
79,01KQ38C4101D6774F2F02331,01KQ38C4101D6774F2F02331_CHUNK_002,Outage Detection & Monitoring,Agent reports initial pipeline failure and del...,negative,0.815866


In [16]:
sample_indices = emb_df.sample(
    min(5, len(emb_df)),
    random_state=42
).index.tolist()

for idx in sample_indices:
    print("=" * 100)
    print("QUERY CHUNK")
    print("chunk_id:", emb_df.loc[idx, "chunk_id"])
    print("label:", emb_df.loc[idx, "chunk_label"])
    print("summary:", emb_df.loc[idx, "chunk_summary"])
    print("\nNEAREST CHUNKS")
    display(show_nearest_chunks(idx, top_k=5))

QUERY CHUNK
chunk_id: 01KQ31ED026F745FE2A55CCD_CHUNK_004
label: Identity Audit Trail
summary: Design identity audit trail schema and Comply integration for compliance reporting.

NEAREST CHUNKS


/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,meeting_id,chunk_id,chunk_label,chunk_summary,chunk_sentiment,similarity
70,01KQ31ED026F745FE2A55CCD,01KQ31ED026F745FE2A55CCD_CHUNK_001,Roadmap Planning,"Team reviews Q2 Identity roadmap, highlighting...",positive,0.790301
40,01KQ1DE954A807A5D2653175,01KQ1DE954A807A5D2653175_CHUNK_003,Compliance Reporting Validation,Reviewed HIPAA and ISO validation issues and p...,neutral,0.779582
48,01KQ2217B066855A3B7814CB,01KQ2217B066855A3B7814CB_CHUNK_006,Compliance Documentation,"Provided audit summary, compliance discussion,...",mixed-negative,0.765736
23,01KQ1267C6AA7D9B3125FEC8,01KQ1267C6AA7D9B3125FEC8_CHUNK_004,Incident Tracking Integration,Identify incident tracking integration gaps an...,negative,0.757046
43,01KQ1DE954A807A5D2653175,01KQ1DE954A807A5D2653175_CHUNK_006,SSO Scope Clarification,Clarified SSO integration scope for Comply v2.,neutral,0.751726


QUERY CHUNK
chunk_id: 01KQ03B0303900521BB089CA_CHUNK_001
label: Outage Remediation
summary: Agent outlines outage status, root cause, and initial remediation plan.

NEAREST CHUNKS


/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


...

/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,meeting_id,chunk_id,chunk_label,chunk_summary,chunk_sentiment,similarity
29,01KQ1A6B7E81B06F4A13B60D,01KQ1A6B7E81B06F4A13B60D_CHUNK_004,Upgrade Execution & Escalation Plan,"Plan upgrade execution, monitoring, and escala...",mixed-negative,0.724502
45,01KQ2217B066855A3B7814CB,01KQ2217B066855A3B7814CB_CHUNK_003,Remediation Plan and Timeline,"Discussed remediation timeline, redundancy imp...",mixed-negative,0.719392
70,01KQ31ED026F745FE2A55CCD,01KQ31ED026F745FE2A55CCD_CHUNK_001,Roadmap Planning,"Team reviews Q2 Identity roadmap, highlighting...",positive,0.710378
16,01KQ0DFE299AC7A74E8022CA,01KQ0DFE299AC7A74E8022CA_CHUNK_003,Audit Planning,"Outlined audit timeline, onboarding steps, bas...",positive,0.703572
41,01KQ1DE954A807A5D2653175,01KQ1DE954A807A5D2653175_CHUNK_004,Documentation Readiness & Scheduling,"Addressed documentation status, scheduling, an...",mixed-negative,0.697097


QUERY CHUNK
chunk_id: 01KQ0CAE7F064EC93F0540CA_CHUNK_002
label: Timeline Risk Assessment
summary: Product requests timeline for Detect reliability; team assesses risk and resource constraints.

NEAREST CHUNKS


/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,meeting_id,chunk_id,chunk_label,chunk_summary,chunk_sentiment,similarity
45,01KQ2217B066855A3B7814CB,01KQ2217B066855A3B7814CB_CHUNK_003,Remediation Plan and Timeline,"Discussed remediation timeline, redundancy imp...",mixed-negative,0.757890
52,01KQ2331EFD78BF3B1CAB747,01KQ2331EFD78BF3B1CAB747_CHUNK_004,SLA Risk & Fix Timeline,Estimated fix timeline; SLA breach risk and po...,negative,0.738994
39,01KQ1DE954A807A5D2653175,01KQ1DE954A807A5D2653175_CHUNK_002,Incident Impact on Comply Delivery,Discussed Detect outage impact on Comply resou...,negative,0.730018
64,01KQ2B4878EC7B3EE5547007,01KQ2B4878EC7B3EE5547007_CHUNK_004,Timeline & Onboarding,Agent reassures about onboarding speed and aud...,positive,0.729203
55,01KQ2331EFD78BF3B1CAB747,01KQ2331EFD78BF3B1CAB747_CHUNK_007,Churn Risk Assessment,Assessed churn risk; Fortex evaluating alterna...,negative,0.726082


In [17]:
duplicate_text_df = emb_df[
    emb_df.duplicated("embedding_text", keep=False)
].sort_values("embedding_text")

duplicate_text_df[
    ["meeting_id", "chunk_id", "chunk_label", "chunk_summary", "embedding_text"]
]

,meeting_id,chunk_id,chunk_label,chunk_summary,embedding_text


In [18]:
emb_df["embedding_dim"] = emb_df["embedding"].apply(
    lambda x: len(x) if isinstance(x, list) else None
)

In [9]:
embedding_quality_summary = {
    "total_chunks": len(emb_df),
    "unique_meetings": emb_df["meeting_id"].nunique(),
    "missing_embeddings": int(emb_df["embedding"].isna().sum()),
    "embedding_dimensions": emb_df["embedding_dim"].value_counts().to_dict(),
    "duplicate_embedding_texts": len(duplicate_text_df),
}

embedding_quality_summary

{'total_chunks': 111,
 'unique_meetings': 20,
 'missing_embeddings': 0,
 'embedding_dimensions': {1024: 111},
 'duplicate_embedding_texts': 0}

In [11]:
show_nearest_chunks(row_index=5)

/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/swatithapaswatithapa/Documents/study/transcript-mining/trans/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,meeting_id,chunk_id,chunk_label,chunk_summary,chunk_sentiment,similarity
3,01KQ03B0303900521BB089CA,01KQ03B0303900521BB089CA_CHUNK_004,Customer Communication,Agent offers to draft updated customer communi...,neutral,0.849227
2,01KQ03B0303900521BB089CA,01KQ03B0303900521BB089CA_CHUNK_003,Remediation Plan,"Agent details redundant nodes, circuit breaker...",neutral,0.836966
102,01KQ351E141926AB7CAB668D,01KQ351E141926AB7CAB668D_CHUNK_002,SLA Review & Renewal,"Agent proposes engineering fixes, SLA review, ...",mixed-negative,0.822529
80,01KQ298F2846EFE59B0D70E2,01KQ298F2846EFE59B0D70E2_CHUNK_003,Post-Mortem & Transparency,"Address ticket spike, legal, transparency, pos...",mixed-negative,0.822407
27,01KQ0F8AFF3DA34FD4580008,01KQ0F8AFF3DA34FD4580008_CHUNK_005,Action Item Recap,Agent recaps action items and next steps.,neutral,0.820054
